# ЛР 02.1 — Глобальная интерпретация моделей (TODO)

## Цель
- взять лучшие неполные набор признаков (feature set) из ЛР 01;
- сравнить глобальные объяснения `LogisticRegression` и `RandomForestClassifier`;
- проверить, как native importance соотносится с `permutation importance`;
- собрать компактную summary по `partial dependence` для числовых признаков.

## Что важно
- интерпретируем не raw-модель, а модель после препроцессинга;
- не путайте importance и причинность;
- фиксируйте не только код, но и смысл различий между объяснениями.

In [ ]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '02-model-interpretability-and-explainability',
    cwd.parent / '02-model-interpretability-and-explainability',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 02 или из корня репозитория.')
spec = importlib.util.spec_from_file_location('lab02_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

## Шаг 1. Загрузка данных и набор признаков (feature set) из ЛР 01

Выбираем лучший **неполный** набор признаков по артефактам первой лабораторной,
чтобы интерпретация не дублировала baseline `full`.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [ ]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

prepared = {}
rows = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_test, y_train, y_test = lab.train_test_split_stratified(x, y)
    feature_set_name = lab.choose_best_nonfull_feature_set(model_results, feature_sets, dataset_name)
    selected_features = feature_sets[dataset_name][feature_set_name]
    prepared[dataset_name] = {
        'x_train': x_train,
        'x_test': x_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_set_name': feature_set_name,
        'selected_features': selected_features,
    }
    rows.append({
        'dataset': dataset_name,
        'feature_set': feature_set_name,
        'n_selected_features': len(selected_features),
        'selected_preview': ', '.join(selected_features[:5]),
    })

selection_summary = pd.DataFrame(rows)
selection_summary

## Шаг 2. Обучение двух моделей на выбранных набор признаков (feature set)

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [ ]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.

artifacts = {}
quality_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, ctx in prepared.items():
    artifacts[dataset_name] = {
        'LogisticRegression': lab.fit_selected_model(
            ctx['x_train'],
            ctx['x_test'],
            ctx['y_train'],
            ctx['y_test'],
            ctx['selected_features'],
            LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
        ),
        'RandomForest': lab.fit_selected_model(
            ctx['x_train'],
            ctx['x_test'],
            ctx['y_train'],
            ctx['y_test'],
            ctx['selected_features'],
            RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=3,
                class_weight='balanced',
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    }

    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in artifacts[dataset_name].items():
        metrics = lab.summarize_model_quality(artifact)
        quality_rows.append({
            'dataset': dataset_name,
            'model': model_name,
            'feature_set': ctx['feature_set_name'],
            **metrics,
        })

quality_summary = pd.DataFrame(quality_rows).sort_values(['dataset', 'roc_auc'], ascending=[True, False])
quality_summary

## Шаг 3. Глобальные объяснения

Собираем единый long-format `global_importance_comparison`:
- `coef_abs` для линейной модели;
- `feature_importance` для дерева;
- `permutation` для обеих моделей.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [ ]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

importance_frames = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, model_map in artifacts.items():
    feature_set_name = prepared[dataset_name]['feature_set_name']
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in model_map.items():
        importance_frames.append(
            lab.build_global_importance_table(
                artifact=artifact,
                dataset_name=dataset_name,
                model_name=model_name,
                feature_set_name=feature_set_name,
            )
        )

global_importance_comparison = pd.concat(importance_frames, ignore_index=True)
top_global = (
    global_importance_comparison
    .sort_values(['dataset', 'model', 'method', 'rank'])
    .groupby(['dataset', 'model', 'method'], group_keys=False)
    .head(8)
)
top_global

## Шаг 4. Partial Dependence для числовых raw-признаков

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [ ]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

curve_frames = []
summary_frames = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, model_map in artifacts.items():
    feature_set_name = prepared[dataset_name]['feature_set_name']
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in model_map.items():
        curve_df, summary_df = lab.build_partial_dependence_summary(
            artifact=artifact,
            dataset_name=dataset_name,
            model_name=model_name,
            feature_set_name=feature_set_name,
        )
        curve_frames.append(curve_df)
        summary_frames.append(summary_df)

partial_dependence_curves = pd.concat(curve_frames, ignore_index=True)
partial_dependence_summary = (
    pd.concat(summary_frames, ignore_index=True)
    .sort_values(['dataset', 'score_delta'], ascending=[True, False])
    .reset_index(drop=True)
)
partial_dependence_summary

## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
- TODO (обязательно): 3-5 предложений о различиях между native importance и permutation importance.
- TODO (обязательно): поясните, почему partial dependence считается модельно-зависимым упрощением.

### Сравнение с альтернативами
- TODO (обязательно): сравните минимум два способа глобальной интерпретации.
- Формат: когда один подход полезнее другого и почему.

### Источники
- TODO (обязательно): минимум 1-2 источника (URL и/или `study-notes/*.md`).

### Глоссарий незнакомых терминов (обязательно)
- TODO (обязательно): добавьте минимум 2-3 новых термина в `study-notes/glossary.md`.
- В конце блока напишите строку: `Глоссарий обновлен: <термин_1>, <термин_2>, ...`.

## Контрольные точки
1. В `global_importance_comparison` есть колонки `dataset, model, feature_set, method, feature, score, rank`.
2. Для каждого датасета есть обе модели.
3. В `partial_dependence_summary` есть минимум по 2-3 raw-признака на датасет.

## Обязательные самостоятельные задания (без образца в solutions)

### Задание 1. Согласованность глобальных объяснений
- Возьмите `global_importance_comparison`.
- Для каждой тройки `dataset + model + method` проверьте top-5 и top-8 признаков.
- Коротко опишите, какие признаки стабильно держатся в топе, а какие зависят от метода.

### Задание 2. Интерпретация partial dependence
- Возьмите `partial_dependence_summary`.
- Найдите признаки с максимальным `score_delta`.
- Поясните, где тренд выглядит монотонным, а где нет.

### Задание 3. Экспорт артефактов и краткий вывод
- Сохраните `outputs/global_importance_comparison.csv`.
- Сохраните `outputs/partial_dependence_summary.csv`.
- Добавьте 2-3 коротких вывода отдельно для `medical` и `finance`.

In [ ]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

# TODO(обязательно):
# 1) Проверьте required columns для global_importance_comparison и partial_dependence_summary.
# 2) Сохраните оба DataFrame в CSV внутри outputs/.
# 3) Добавьте текстовую интерпретацию в markdown-блоки выше.

raise NotImplementedError(
    'Самостоятельный блок не завершен: сохраните outputs/global_importance_comparison.csv и outputs/partial_dependence_summary.csv.'
)
